### Install dependencies

In [ ]:
# !pip install torch lightrag-hku transformers sentence-transformers ragas datasets python-dotenv accelerate langchain-community nest_asyncio
# !pip install "langchain-community<0.2.0"
# !pip install --upgrade ragas langchain langchain-core langchain-community langchain-openai
# !pip install -e .

Obtaining file:///Users/quanninh/Documents/Fourth_year/Project/HealthCareAgent
ERROR: file:///Users/quanninh/Documents/Fourth_year/Project/HealthCareAgent does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [11]:
import os

# The exact path to the broken file from your error log
file_path = "/opt/anaconda3/envs/HealthCare/lib/python3.11/site-packages/ragas/llms/base.py"

# Read the file
with open(file_path, "r") as f:
    content = f.read()

# Replace the strict import with a safe try-except block
broken_import = "from langchain_community.chat_models.vertexai import ChatVertexAI"
safe_import = """try:
    from langchain_community.chat_models.vertexai import ChatVertexAI
except ImportError:
    ChatVertexAI = None"""

if broken_import in content:
    content = content.replace(broken_import, safe_import)
    
    # Write the fix back to the file
    with open(file_path, "w") as f:
        f.write(content)
    print("✅ Successfully patched ragas/llms/base.py!")
else:
    print("⚠️ Patch already applied or import not found.")

✅ Successfully patched ragas/llms/base.py!


### Import libraries 

In [1]:
import os
import json
import glob
import asyncio
import re
from pathlib import Path
from typing import Optional

import torch
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from sentence_transformers import SentenceTransformer

from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from lightrag.llm.openai import openai_complete_if_cache


/opt/anaconda3/envs/HealthCare/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
PAYLOAD_DIR = "payload_for_ref/payloads"
DATASET_ROOT = "medical_datasets/whole_multicare_dataset"
WORKING_DIR = "./lightrag_medgemma_wd"
HF_TOKEN = os.getenv("HF_TOKEN")
MAX_CASES = 100
EVAL_SAMPLES = 10
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MEDGEMMA_MODEL = "google/medgemma-4b-it"

os.makedirs(WORKING_DIR, exist_ok=True)

### Process data

In [3]:
"""Reconstruct image path from .json files"""
def remap_image_path(broken_path: str, dataset_root: str) -> str:
    fname = os.path.basename(broken_path)         
    prefix4 = fname[:4]                            
    prefix5 = fname[:5]                            
    return os.path.join(dataset_root, prefix4, prefix5, fname)

def parse_payload(json_path: str, dataset_root: str) -> dict:
    with open(json_path, "r", encoding="utf_8") as f:
        blocks = json.load(f)

    texts = []
    images = []
    for block in blocks:
        if block.get("type") == "text":
            texts.append(block.get("text", ""))
        elif block.get("type") == "image":
            broken = block.get("img_path", "")
            correct_path = remap_image_path(broken, dataset_root) if broken else ""
            images.append({
                "path": correct_path,
                "caption": " ".join(block.get("image_caption", [])),
                "footnote": " ".join(block.get("image_footnote", []))
            })

    full_text = (os.linesep * 2).join(texts)

    for img in images:
        if img["caption"]:
            full_text += os.linesep + f"[Image] {img['caption']}"
        if img["footnote"]:
            full_text += os.linesep + f"[ImageNote] {img['footnote']}"

    return {"case_id": Path(json_path).stem, "text": full_text, "images": images}

def load_all_payloads(payload_dir: str, dataset_root: str, max_cases: int = MAX_CASES):
    files = sorted(glob.glob(os.path.join(payload_dir, "*.json")))[:max_cases]
    print(f"Loading {len(files)} payload files...")
    cases = [parse_payload(f, dataset_root) for f in files]
    print(f"Loaded {len(cases)} cases.")
    return cases

def load_eval_case(payload_dir: str, dataset_root: str, max_cases: int = MAX_CASES):
    files = sorted(glob.glob(os.path.join(payload_dir, "*.json")))[max_cases:max_cases+max_cases]
    print(f"Loading {len(files)} payload files...")
    eval_cases = [parse_payload(f, dataset_root) for f in files]
    print(f"Loaded {len(cases)} cases.")
    return eval_cases

### MedGemmaWrapper

In [ ]:
# class MedGemmaWrapper:
#     def __init__(self, model_id: str = MEDGEMMA_MODEL, hf_token: Optional[str] = None):
#         print(f"Loading MedGemma model.....")
#         device = (
#             "cuda" if torch.cuda.is_available() else
#             "mps" if torch.backends.mps.is_available() else
#             "cpu"
#         )
#         dtype = torch.bfloat16 if device in ("cuda", "mps") else torch.float32

#         self.tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token, trust_remote_code=True)
#         self.model = AutoModelForCausalLM.from_pretrained(model_id, token=hf_token, torch_dtype=dtype, device_map="auto" if device == "cuda" else None, trust_remote_code=True).to(device)
#         self.device = device

#     def generate(self, prompt: str, max_new_tokens: int = 512) -> str:
#         inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
#         with torch.no_grad():
#             outputs = self.model.generate(
#                 **inputs,
#                 max_new_tokens=max_new_tokens,
#                 do_sample=False,
#                 temperature=None,
#                 top_p=None
#             )

#         new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
#         return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

# # Declare medgemma_wrapper object
# # medgemma_wrapper = MedGemmaWrapper(model_id=MEDGEMMA_MODEL, hf_token=HF_TOKEN)

# _llm_lock = asyncio.Lock()
# async def medgemma_llm_func(prompt, system_prompt=None, history_messages=[], **kwarg) -> str:
#     strict_rules = (
#         "\n\nCRITICAL FORMATTING INSTRUCTIONS:\n"
#         "- DO NOT wrap entity names, types, or descriptions in angle brackets (< or >).\n"
#         "- You must output EXACTLY the requested fields (4 for Entities, 5 for Relations).\n"
#         "- Output ONLY the requested format and nothing else."
#     )

#     prompt = prompt + strict_rules

#     full_prompt = ""
#     if system_prompt:
#         full_prompt += f"<system>" + os.linesep + f"{system_prompt}" + os.linesep + f"</system>" + os.linesep * 2
#     for msg in history_messages:
#         role = msg.get("role", "user")
#         content  = msg.get("content", "")
#         full_prompt += f"<{role}>" + os.linesep + f"{content}" + os.linesep + f"</{role}>" + os.linesep * 2
#     full_prompt += f"<user>" + os.linesep + f"{prompt}" + os.linesep + f"</user>" + os.linesep + f"<model>" + os.linesep

#     loop = asyncio.get_event_loop()
#     async with _llm_lock:
#         loop = asyncio.get_event_loop()
#         result = await loop.run_in_executor(None, medgemma_wrapper.generate, full_prompt)
#     if result:
#         result = result.replace("<", "").replace(">", "")
#     return result



### Embedding Function

In [4]:
embed_model = SentenceTransformer(EMBEDDING_MODEL)
_embed_lock = asyncio.Lock()  

async def local_embedding_func(texts: list[str]) -> list[list[float]]:
    async with _embed_lock:
        loop = asyncio.get_event_loop()
        embeddings = await loop.run_in_executor(
            None, lambda: embed_model.encode(texts, normalize_embeddings=True)
        )
    return embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11818.50it/s]


### OpenAIWrapper

In [17]:
async def ollama_openai_wrapper(prompt, system_prompt=None, history_messages=[], **kwargs):
    strict_rule = "\n\nYou are a medical AI assistant. Answer the user's question STRICTLY and ONLY using the provided context. If the context does not contain the answer, say 'I do not have enough information to answer this based on the provided text.' Do not use outside knowledge."
    
    if system_prompt:
        system_prompt += strict_rule
    else:
        system_prompt = strict_rule
        
    return await openai_complete_if_cache(
        model="llama3.1",
        prompt=prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key="ollama",
        base_url="http://localhost:11434/v1",
        **kwargs
    )


### Build LightRAG

In [18]:
async def build_lightrag(cases: list[dict]) -> LightRAG:
    rag = LightRAG(
        working_dir=WORKING_DIR,
        llm_model_func=ollama_openai_wrapper,
        embedding_func=EmbeddingFunc(embedding_dim=384, max_token_size=512, func=local_embedding_func)
    )

    await rag.initialize_storages()
    
    print(f"Inserting {len(cases)} cases into LightRAG knowledge graph...")
    for i, case in enumerate(cases):
        if case["text"].strip():
            await rag.ainsert(case["text"])
            if (i + 1) % 10 == 0:
                print(f" Inserted {i + 1}/{len(cases)} cases")

    print("Knowledge graph built successfully")
    return rag

async def load_existing_rag() -> LightRAG:
    print(f"Load existing LightRAG from {WORKING_DIR}...")

    rag = LightRAG(
            working_dir=WORKING_DIR,
            llm_model_func=ollama_openai_wrapper,
            embedding_func=EmbeddingFunc(embedding_dim=384, max_token_size=512, func=local_embedding_func)
        )

    await rag.initialize_storages()
    return rag

async def query_rag(rag: LightRAG, question: str, mode: str = "hybrid") -> dict:
    result = await rag.aquery(question, param=QueryParam(mode=mode))
    return {"question": question, "answer": result}



### Evaluation Metric

In [ ]:
async def evaluate_retrieval(cases: list, rag, k: int = 5, n_samples: int = 10):
    eval_data = []
    for case in cases[:n_samples]:
        # 1. Generate the question
        prompt = f"""
        Based ONLY on the following clinical text, write exactly ONE realistic medical question 
        that a doctor might ask, which can be answered by this text. Do not output anything else.
        Text: {case['text'][:1500]}
        """
        realistic_question = await ollama_openai_wrapper(prompt=prompt)
        realistic_question = realistic_question.strip()
        # 2. QUERY THE ACTUAL VECTOR DATABASE
        vdb_results = await rag.chunks_vdb.query(realistic_question, top_k=k)
        
        # 3. FIX: EXTRACT THE RAW TEXT CONTENT instead of the ID
        # (NanoVectorDB usually stores the text inside 'content' or 'text' key)
        retrieved_texts = [res.get('content', res.get('text', '')) for res in vdb_results]
        eval_data.append({
            "ground_truth_text": case["text"],
            "retrieved_texts": retrieved_texts, 
        })
    # 4. CALCULATE THE METRICS 
    recall_at_k_count = 0
    top1_accuracy_count = 0
    mrr_sum = 0.0
    n = len(eval_data)
    for item in eval_data:
        gt_text = item["ground_truth_text"]
        top_k_retrieved_texts = item["retrieved_texts"][:k]
        
        # We check if a significant portion of the ground truth text (e.g. first 50 chars) 
        # is inside the retrieved chunk, or vice versa, to confirm a match.
        match_substring = gt_text[:50].strip()
        # Check Top-1
        if len(top_k_retrieved_texts) > 0 and (match_substring in top_k_retrieved_texts[0] or top_k_retrieved_texts[0][:50] in gt_text):
            top1_accuracy_count += 1
        # Check Recall and MRR
        for rank_idx, retrieved_text in enumerate(top_k_retrieved_texts):
            if match_substring in retrieved_text or retrieved_text[:50] in gt_text:
                recall_at_k_count += 1
                rank = rank_idx + 1
                mrr_sum += 1.0 / rank
                break 
    print(f"Total Queries (n): {n}")
    print(f"Cutoff (k): {k}")
    print(f"Top-1 Accuracy: {top1_accuracy_count / n:.4f}")
    print(f"Recall@{k}: {recall_at_k_count / n:.4f}")
    print(f"MRR@{k}: {mrr_sum / n:.4f}")
    return {"recall": recall_at_k_count / n, "mrr": mrr_sum / n}

In [ ]:
# async def evaluate_retrieval(cases: list, k: int = 5, n_samples: int = 10):
#     case_texts = [c["text"] for c in cases]
#     case_ids = [c["case_id"] for c in cases]
#     doc_embeddings = await local_embedding_func(case_texts)

#     eval_data = []

#     for case in cases[:n_samples]:
#         prompt = f"""
#         Based ONLY on the following clinical text, write exactly ONE realistic medical question 
#         that a doctor might ask, which can be answered by this text. Do not output anything else.
#         Text: {case['text'][:1500]}
#         """

#         realistic_question = await ollama_openai_wrapper(prompt=prompt)
#         realistic_question = realistic_question.strip()

#         print(f"Generated Q: {realistic_question[:80]}...")

#         query_embedding = await local_embedding_func([realistic_question])

#         similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

#         # Ranked documents
#         ranked_indices = np.argsort(similarities)[::-1]
#         ranked_case_ids = [case_ids[idx] for idx in ranked_indices]

#         eval_data.append({
#             "ground_truth_id": case["case_id"],
#             "retrieved_ids": ranked_case_ids, 
#         })

#     recall_at_k_count = 0
#     top1_accuracy_count = 0
#     mrr_sum = 0.0
#     n = len(eval_data)

#     for item in eval_data:
#         gt_id = item["ground_truth_id"]
#         top_k_retrieved = item["retrieved_ids"][:k]

#         if len(top_k_retrieved) > 0 and top_k_retrieved[0] == gt_id:
#             top1_accuracy_count += 1

#         if gt_id in top_k_retrieved:
#             recall_at_k_count += 1
#             rank = top_k_retrieved.index(gt_id) + 1
#             mrr_sum += 1.0 / rank

#     print(f"Total Queries (n): {n}")
#     print(f"Cutoff (k): {k}")
#     print(f"Top-1 Accuracy: {top1_accuracy_count / n:.4f} (How often the exact document was #1)")
#     print(f"Recall@{k}: {recall_at_k_count / n:.4f} (How often the document was in the top {k})")
#     print(f"MRR@{k}: {mrr_sum / n:.4f} (Average rank quality)")

#     return {"recall": recall_at_k_count / n, "mrr": mrr_sum / n}

async def evaluate_rag(cases: list, rag, n_samples: int = 5):
    total_score = 0
    valid_evals = 0

    for case in cases[:n_samples]:
        prompt_q = f"Based ONLY on this text, write one realistic medical question: {case['text'][:1000]}"
        question = await ollama_openai_wrapper(prompt=prompt_q)
        question = question.strip()

        print(f"\nQ: {question}...")

        # Evaluate LightRAG
        rag_response = await query_rag(rag, question, mode="hybrid")
        lightrag_answer = rag_response['answer'] if rag_response and rag_response.get('answer') else "No answer."

        # Use LLM to evaluate answers
        eval_prompt = f"""
        You are a medical grader.
        GROUND TRUTH TEXT: {case['text'][:1500]}
        QUESTION: {question}
        LIGHTRAG'S ANSWER: {lightrag_answer}
        
        Grade LightRAG's answer from 1 to 10 on Accuracy. 
        Did LightRAG successfully use the graph to find the right answer?
        Output ONLY a single integer from 1 to 10.
        """

        try:
            score_response = await ollama_openai_wrapper(prompt=eval_prompt)
            match = re.search(r'\b(10|[1-9])\b', score_response)
            if match:
                score = int(match.group(0))
                total_score += score
                valid_evals += 1
                print(f"-> LightRAG Score: {score}/10")
            else:
                print(f"-> Failed to parse score")
        except Exception as e:
            print(f"-> Error: {e}")

    avg_score = (total_score / valid_evals) if valid_evals > 0 else 0
    print("\n===== FINAL LIGHTRAG GRAPH METRICS =====")
    print(f"LightRAG Hybrid Accuracy: {avg_score:.2f} / 10.0")
    
    return avg_score



### To build Light RAG

In [7]:
cases = load_all_payloads(PAYLOAD_DIR, DATASET_ROOT, max_cases=MAX_CASES)
# rag = await build_lightrag(cases)

Loading 100 payload files...
Loaded 100 cases.


### Print evaluation metrics

In [8]:
# eval_cases = load_eval_case(PAYLOAD_DIR, DATASET_ROOT, max_cases=MAX_CASES)
rag_instance = await load_existing_rag()
# eval_results = await evaluate_retrieval(cases, rag_instance)
# avg_score = await evaluate_rag(cases, rag_instance)

Load existing LightRAG from ./lightrag_medgemma_wd...


INFO: [] Loaded graph from ./lightrag_medgemma_wd/graph_chunk_entity_relation.graphml with 3404 nodes, 2180 edges
INFO:nano-vectordb:Load (3404, 384) data
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': './lightrag_medgemma_wd/vdb_entities.json'} 3404 data
INFO:nano-vectordb:Load (2180, 384) data
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': './lightrag_medgemma_wd/vdb_relationships.json'} 2180 data
INFO:nano-vectordb:Load (295, 384) data
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': './lightrag_medgemma_wd/vdb_chunks.json'} 295 data
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=4, timeout=240
INFO:  - keyword: None/None, host=None, max_async=4, timeout=240
INFO:  - query: None/None, host=None, max_async=4, timeout=240
INFO:  - vlm: None/None, host=None, max_async=4, timeout=240
INFO: [] Process 8484 KV load full_docs with 100 reco

## Ragas evaluation

In [24]:
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

# 1. Create a standard LangChain ChatOpenAI object, but point it to Ollama
langchain_llm = ChatOpenAI(
    model="llama3.1",
    base_url="http://localhost:11434/v1", 
    api_key="ollama",
    temperature=0
)

# 2. Tell Ragas to wrap this LangChain model and use it as the judge
ragas_llm = LangchainLLMWrapper(langchain_llm)

/var/folders/95/9hrnbtwd08sf4g5dv_mty4380000gn/T/ipykernel_8484/1885950405.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(langchain_llm)


In [13]:
# Create lists to store the evaluation data
questions = []
answers = []
contexts = []
ground_truths = []

for case in cases[:5]:  # Just run 5 for a quick test
    # 1. Generate the question (using your existing code)
    prompt_q = f"Based ONLY on this text, write one realistic medical question. Output ONLY the question itself, with no introductory text or conversational filler: {case['text'][:1000]}"
    question = await ollama_openai_wrapper(prompt=prompt_q)
    question = question.strip()
    
    # 2. Get the answer from LightRAG
    rag_response = await query_rag(rag_instance, question, mode="hybrid")
    
    # 3. Get the retrieved text chunks (you did this in your evaluate_retrieval function!)
    vdb_results = await rag_instance.chunks_vdb.query(question, top_k=5)
    retrieved_texts = [res.get('content', res.get('text', '')) for res in vdb_results]

    # Save them to our lists
    questions.append(question)
    answers.append(rag_response['answer'])
    contexts.append(retrieved_texts)
    ground_truths.append(case['text'][:1500]) # Use the raw clinical text as ground truth

INFO:  == LLM cache == saving: hybrid:keywords:d2b64dd1cfcb05c44a8e63b716c79498
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.63it/s]
INFO: Query nodes: positive, significance, patients (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 378 relations
INFO: Query edges: COVID-19, autoantibodies, myositis (top_k:40, cosine:0.2)
INFO: Global query: 52 entites, 40 relations
INFO: Raw search results: 91 entities, 413 relations, 0 vector chunks
INFO: After truncation: 91 entities, 188 relations
INFO: Selecting 209 from 209 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 188 relations
INFO: Round-robin merged chunks: 209 -> 209 (deduplicated 0)
INFO: Final context: 91 entities, 188 relations, 20 chunks
INFO: Final chunks S+F/O: E2/1 E4/2 E2/3 E3/4 E2/5 E4/6 E4/7 E1/8 E3/9 E3/10 E3/11 E1/12 E2/13 E2/14 E4/15 E5/16 E1/17 E2/18 E1/19 E3/20
INFO:  == LLM cache == saving: hybrid:query:2347c0b388ced23d993012810724b0f1
Batches: 100%|██████████| 1

In [28]:
from datasets import Dataset
from ragas import evaluate

# 1. Ignore the warnings and use the standard, working metric instances
from ragas.metrics import faithfulness, answer_relevancy

# 2. Setup the run configuration to prevent Ollama timeouts
from ragas.run_config import RunConfig
run_config = RunConfig(timeout=600, max_workers=1)

# 3. Create the LLM and Embeddings using LangChain
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper

# Create standard LangChain objects
langchain_llm = ChatOpenAI(
    model="llama3.1",
    base_url="http://localhost:11434/v1", 
    api_key="ollama",
    temperature=0
)
ragas_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Wrap the LLM for Ragas
ragas_llm = LangchainLLMWrapper(langchain_llm)

# 4. Load your dataset
eval_dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
})

# 5. Evaluate! (Passing everything globally)
results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy], 
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=run_config
)

df_results = results.to_pandas()
display(df_results)

/var/folders/95/9hrnbtwd08sf4g5dv_mty4380000gn/T/ipykernel_8484/2580277019.py:5: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
/var/folders/95/9hrnbtwd08sf4g5dv_mty4380000gn/T/ipykernel_8484/2580277019.py:5: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy
INFO:sentence_transformers.base.model:No device provided, using mps
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6612.68it/s]
/var/folders/95/9hrnbtwd08sf

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy
0,What is the significance of positive myositis-...,[=== CLINICAL CASE REPORT ===\nTitle: Positive...,The documents provided do not directly answer ...,=== CLINICAL CASE REPORT ===\nTitle: Positive ...,0.50000,0.000000
1,What is the significance of the patient's norm...,"[ SARS-CoV-2 (>50%), hemogram with neutrophili...",The patient's normal peripheral oxygen saturat...,=== CLINICAL CASE REPORT ===\nTitle: Positive ...,0.70000,0.914194
2,What is the potential benefit-to-risk ratio of...,[=== CLINICAL CASE REPORT ===\nTitle: A Rare C...,"Based on the information provided, it is not c...",=== CLINICAL CASE REPORT ===\nTitle: A Rare Ca...,0.26087,0.000000
3,What are the potential microbiological consequ...,[=== CLINICAL CASE REPORT ===\nTitle: Bloodstr...,There is no mention of Ralstonia manitolfilyti...,=== CLINICAL CASE REPORT ===\nTitle: Bloodstre...,0.87500,0.000000
4,What is the long-term prognosis for lung funct...,[=== CLINICAL CASE REPORT ===\nTitle: Full-Ter...,The long-term prognosis for lung function in a...,=== CLINICAL CASE REPORT ===\nTitle: Full-Term...,NaN,1.000000
